# Structure-aware Table QA with Qwen3 + LoRA

This notebook runs a controlled experiment ladder:

1. `serialized_table_lora`: standard leading-crop serialized table baseline.
2. `serialized_retrieval_lora`: adds deterministic question-conditioned row/column selection.
3. `structured_2d_lora`: preserves lexical table tokens and adds learned row, column, and header/data embeddings.

All three train on complete answer sets, use official WikiTableQuestions denotation accuracy for early stopping, and save to separate Drive directories. Run the baseline first, then the proposed model. The retrieval-only configuration is the attribution ablation.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
%cd /content/table-cnn-mrc

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

DRIVE_ROOT = Path("/content/drive/MyDrive/cnn_qwen_table_mcr/outputs")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Drive output root: {DRIVE_ROOT}")

## Choose one experiment

Start with `serialized_table_lora`. After it finishes, change the value to `structured_2d_lora` and rerun from this cell. Use `serialized_retrieval_lora` afterward if you want the clean retrieval ablation. Each experiment resumes independently after a disconnect.

In [ ]:
EXPERIMENT = "serialized_table_lora"
# EXPERIMENT = "serialized_retrieval_lora"
# EXPERIMENT = "structured_2d_lora"

VALID_EXPERIMENTS = {
    "serialized_table_lora",
    "serialized_retrieval_lora",
    "structured_2d_lora",
}
assert EXPERIMENT in VALID_EXPERIMENTS
CONFIG_PATH = REPO_DIR / "configs" / f"{EXPERIMENT}.yaml"
DRIVE_OUTPUT = DRIVE_ROOT / EXPERIMENT
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Config: {CONFIG_PATH}")
print(f"Persistent output: {DRIVE_OUTPUT}")

## Smoke test

This verifies a real WTQ example, answer-only loss, gradient flow through LoRA and any structural adapters, and deterministic generation.

In [ ]:
%cd {REPO_DIR}
!PYTHONUNBUFFERED=1 python -u scripts/smoke_test.py --config "{CONFIG_PATH}"

## Train or resume

The first run downloads and caches official WTQ evaluator metadata under the Drive diagnostics directory. Checkpoints are mirrored every 100 optimizer steps and every epoch. Rerunning this cell resumes the latest full checkpoint.

In [ ]:
%cd {REPO_DIR}
!PYTHONUNBUFFERED=1 python -u scripts/run_experiment.py \
    --config "{CONFIG_PATH}" \
    --mirror-output-dir "{DRIVE_OUTPUT}"

## Compare completed runs

In [ ]:
import json
import pandas as pd
from IPython.display import display

rows = []
for name in sorted(VALID_EXPERIMENTS):
    history_path = DRIVE_ROOT / name / "history.json"
    if not history_path.is_file():
        continue
    with history_path.open() as handle:
        history = json.load(handle)
    rows.append({
        "experiment": name,
        "status": history.get("status"),
        "epochs": len(history.get("epochs", [])),
        "primary_metric": history.get("primary_metric"),
        "best_metric": history.get("best_metric"),
        "final_training_loss": history.get("epochs", [{}])[-1].get("training_loss") if history.get("epochs") else None,
    })
display(pd.DataFrame(rows).sort_values("best_metric", ascending=False) if rows else pd.DataFrame())

## Audit the best checkpoint

This performs a 200-example correct-versus-shuffled table test with official denotation scoring and saves representative predictions.

In [ ]:
AUDIT_OUTPUT = DRIVE_OUTPUT / "diagnostics/checkpoint_audit"
%cd {REPO_DIR}
!PYTHONUNBUFFERED=1 python -u scripts/audit_saved_checkpoint.py \
    --config "{CONFIG_PATH}" \
    --run-dir "{DRIVE_OUTPUT}" \
    --checkpoint best \
    --max-examples 200 \
    --sample-count 5 \
    --output-dir "{AUDIT_OUTPUT}"